In [2]:
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from torchvision.transforms import InterpolationMode
from torchvision import transforms

OSError: [WinError 4551] 애플리케이션 제어 정책에서 이 파일을 차단했습니다. Error loading "c:\Users\taehyeok\miniconda3\envs\taehyeok\lib\site-packages\torch\lib\c10_cuda.dll" or one of its dependencies.

In [4]:
class Cutout:
    def __init__(self, num_holes=4, cut_ratio=0.2, fill=0):
        self.num_holes = num_holes
        self.cut_ratio = cut_ratio
        self.fill = fill

    def __call__(self, img):
        arr = np.asarray(img, dtype=np.uint8).copy()
        h, w = arr.shape[:2]
        if h == 0 or w == 0:
            return img

        cut_size = max(1, int(min(h, w) * self.cut_ratio))
        half = cut_size // 2

        for _ in range(max(0, self.num_holes)):
            cy = np.random.randint(0, h)
            cx = np.random.randint(0, w)

            y1 = max(0, cy - half)
            y2 = min(h, cy + half)
            x1 = max(0, cx - half)
            x2 = min(w, cx + half)
            arr[y1:y2, x1:x2] = self.fill

        return Image.fromarray(arr)


class MaskedBernoulliNoiseFast:
    def __init__(self, noise=0.05, min_=0, max_=1):
        self.noise = noise
        self.min_ = min_
        self.max_ = max_

    def __call__(self, img):
        arr = np.asarray(img, dtype=np.uint8)
        h, w = arr.shape[:2]
        if h == 0 or w == 0:
            return img

        wafer_mask = arr > 0
        bernoulli_mask = np.random.rand(h, w) < self.noise
        mask = wafer_mask & bernoulli_mask

        out = arr.copy()
        noise_value = 1 + np.random.randint(self.min_, self.max_ + 1, size=(h, w))
        out[mask] = noise_value[mask].astype(np.uint8)
        return Image.fromarray(out)

In [ ]:

def show_wbm(ax, arr, title=None):
    cmap = ListedColormap(["white", "lightgray", "red"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
    ax.imshow(arr, cmap=cmap, norm=norm, interpolation="nearest")
    if title is not None:
        ax.set_title(title, fontsize=8)
    ax.axis("off")


labeled_path = "./data/wm811k/preprocessing/labeled.pkl"
unlabeled_path = "./data/wm811k/preprocessing/unlabeled.pkl"

use_labeled = True
start_index = 0
num_samples = 8
image_size = 96
np.random.seed(42)

if use_labeled:
    df = pd.read_pickle(labeled_path)
else:
    df = pd.read_pickle(unlabeled_path)

resize_tf = transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.NEAREST)
cutout_tf = Cutout(num_holes=4, cut_ratio=0.2, fill=0)
noise_tf = MaskedBernoulliNoiseFast(noise=0.05, min_=0, max_=1)

indices = list(range(start_index, min(start_index + num_samples, len(df))))
if len(indices) == 0:
    raise ValueError("No samples to visualize. Check start_index/num_samples.")

rows = len(indices)
fig, axes = plt.subplots(rows, 4, figsize=(14, 2.8 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

col_titles = ["Original", "Cutout", "Noise", "Cutout + Noise"]
for c, t in enumerate(col_titles):
    axes[0, c].set_title(t, fontsize=12)

for r, idx in enumerate(indices):
    wafer = df.iloc[idx]["waferMap"]
    img = Image.fromarray(wafer.astype(np.uint8))
    img_resized = resize_tf(img)

    original = np.asarray(img_resized, dtype=np.uint8)
    cutout_only = np.asarray(cutout_tf(img_resized), dtype=np.uint8)
    noise_only = np.asarray(noise_tf(img_resized), dtype=np.uint8)
    cutout_noise = np.asarray(noise_tf(cutout_tf(img_resized)), dtype=np.uint8)

    variants = [original, cutout_only, noise_only, cutout_noise]
    for c, arr in enumerate(variants):
        ax = axes[r, c]
        show_wbm(ax, arr)
        if c == 0:
            ax.set_ylabel(f"idx {idx}", rotation=0, labelpad=24, va="center")

plt.tight_layout()
plt.show()

NameError: name 'transforms' is not defined